# Starbucks Strategic Location Insights: From Analysis to Action

This notebook answers the **"So What?"** question — translating spatial analysis into concrete business recommendations.

Using the Location Fitness Score (LFS) model and Manhattan store data, we identify:
1. **Top expansion opportunities** — underserved high-demand areas
2. **Closure risk candidates** — oversaturated low-demand locations
3. **Competitive white space** — areas where Starbucks could dominate

**Series context:** This is the capstone of a [10-notebook series](https://www.kaggle.com/code/shiratoriseto/manhattan-cafe-wars-starbucks-vs-1200-competitors) combining spatial analysis with NLP on 30 years of Starbucks SEC filings.

## Section 0 — Setup & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import joblib
from pathlib import Path

plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})

# ── Starbucks brand colors ───────────────────────────────────────────
SB_GREEN = "#00704A"
SB_DARK  = "#1E3932"
SB_LIGHT = "#D4E9E2"

# ── path resolution ──────────────────────────────────────────────────
ON_KAGGLE = Path("/kaggle/working").exists()

if ON_KAGGLE:
    _data_candidates = [
        Path("/kaggle/input/manhattan-cafe-wars"),
        Path("/kaggle/input/datasets/shiratoriseto/manhattan-cafe-wars"),
    ]
    DATA_DIR = next((p for p in _data_candidates if p.exists()), None)
    if DATA_DIR is None:
        raise FileNotFoundError("Dataset not found")

    _model_candidates = [
        Path("/kaggle/input/models/shiratoriseto/starbucks-location-fitness-model/other/default/1/lfs_model.joblib"),
        Path("/kaggle/input/starbucks-location-fitness-model/other/default/1/lfs_model.joblib"),
    ]
    MODEL_PATH = next((p for p in _model_candidates if p.exists()), None)
    if MODEL_PATH is None:
        import os
        avail = [os.path.join(r, f) for r, _, fs in os.walk("/kaggle/input") for f in fs if f.endswith(".joblib")]
        raise FileNotFoundError(f"Model not found. Available: {avail}")
else:
    DATA_DIR = Path("../../dataset-upload/v3")
    MODEL_PATH = Path("lfs_model.joblib")

df = pd.read_csv(DATA_DIR / "stores_enriched_v4.csv")
model = joblib.load(MODEL_PATH)
print(f"Loaded {df.shape[0]} stores × {df.shape[1]} features")
print(f"Model: {type(model).__name__} ({model.n_features_in_} features)")

## Section 1 — Market Overview

In [ ]:
lfs = df["location_fitness_score"].dropna()

# ── summary stats ────────────────────────────────────────────────────
print(f"Total stores: {len(df)}")
print(f"Stores with LFS: {len(lfs)}")
print(f"LFS — mean: {lfs.mean():.3f}, median: {lfs.median():.3f}, std: {lfs.std():.3f}")
print(f"LFS — min: {lfs.min():.3f}, max: {lfs.max():.3f}")

# ── LFS distribution with zones ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

q33 = lfs.quantile(0.33)
q67 = lfs.quantile(0.67)

bins = np.linspace(lfs.min(), lfs.max(), 30)
ax.hist(lfs[lfs <= q33], bins=bins, color="#D32F2F", alpha=0.7, label=f"Low (≤{q33:.2f})")
ax.hist(lfs[(lfs > q33) & (lfs <= q67)], bins=bins, color="#FFA726", alpha=0.7, label=f"Medium ({q33:.2f}–{q67:.2f})")
ax.hist(lfs[lfs > q67], bins=bins, color=SB_GREEN, alpha=0.7, label=f"High (>{q67:.2f})")

ax.axvline(lfs.median(), color="black", linestyle="--", linewidth=1, label=f"Median ({lfs.median():.2f})")
ax.set_xlabel("Location Fitness Score")
ax.set_ylabel("Number of Stores")
ax.set_title("Manhattan Starbucks — Location Fitness Score Distribution")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nZone breakdown:")
print(f"  Low  (closure risk):  {(lfs <= q33).sum()} stores ({(lfs <= q33).mean()*100:.0f}%)")
print(f"  Medium (stable):      {((lfs > q33) & (lfs <= q67)).sum()} stores")
print(f"  High (strong fit):    {(lfs > q67).sum()} stores ({(lfs > q67).mean()*100:.0f}%)")

## Section 2 — Top 5 Expansion Opportunities

We identify **underserved high-demand areas** — locations where demand indicators are strong but Starbucks presence is thin.

In [ ]:
# ── expansion score: high demand + low Starbucks density ─────────────
df_valid = df.dropna(subset=["location_fitness_score", "mta_avg_daily_ridership",
                              "nearest_starbucks_dist_m", "n_starbucks_500m"]).copy()

def norm(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

df_valid["expansion_score"] = (
    norm(df_valid["mta_avg_daily_ridership"]) * 0.3 +
    norm(df_valid["ped_count_nearest"].fillna(0)) * 0.2 +
    norm(df_valid["nearest_starbucks_dist_m"]) * 0.25 +
    (1 - norm(df_valid["n_starbucks_500m"])) * 0.25
)

top_expand = df_valid.nlargest(5, "expansion_score")

print("Top 5 Expansion Opportunities:")
print("=" * 90)
for rank, (idx, row) in enumerate(top_expand.iterrows(), 1):
    print(f"\n#{rank} — Lat: {row['lat']:.5f}, Lon: {row['lon']:.5f}")
    print(f"   Subway ridership: {row['mta_avg_daily_ridership']:,.0f}/day")
    print(f"   Nearest Starbucks: {row['nearest_starbucks_dist_m']:.0f}m")
    print(f"   Starbucks within 500m: {row['n_starbucks_500m']:.0f}")
    print(f"   Pedestrian count: {row['ped_count_nearest']:,.0f}")
    print(f"   LFS: {row['location_fitness_score']:.3f}  |  Expansion score: {row['expansion_score']:.3f}")

In [ ]:
# ── expansion opportunity map ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 12))

ax.scatter(df_valid["lon"], df_valid["lat"], c=SB_LIGHT, s=20,
           edgecolors="grey", linewidth=0.3, alpha=0.5, label="Existing stores")

ax.scatter(top_expand["lon"], top_expand["lat"], c="#D4AF37", s=200,
           edgecolors="black", linewidth=1.5, marker="*", zorder=5,
           label="Top 5 expansion opportunities")

for rank, (idx, row) in enumerate(top_expand.iterrows(), 1):
    ax.annotate(f"#{rank}", (row["lon"], row["lat"]),
                textcoords="offset points", xytext=(8, 8),
                fontsize=9, fontweight="bold")

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Expansion Opportunities — Underserved High-Demand Areas")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## Section 3 — Closure Risk Assessment

Stores with **low LFS** face oversaturation (high supply) and/or weak demand — potential candidates for consolidation.

In [ ]:
# ── closure risk: low LFS + high local competition + low ridership ──
df_valid["closure_risk"] = (
    (1 - norm(df_valid["location_fitness_score"])) * 0.4 +
    norm(df_valid["n_starbucks_500m"]) * 0.3 +
    (1 - norm(df_valid["mta_avg_daily_ridership"])) * 0.3
)

top_risk = df_valid.nlargest(5, "closure_risk")

print("Top 5 Closure Risk Candidates:")
print("=" * 90)
for rank, (idx, row) in enumerate(top_risk.iterrows(), 1):
    print(f"\n#{rank} — Lat: {row['lat']:.5f}, Lon: {row['lon']:.5f}")
    print(f"   LFS: {row['location_fitness_score']:.3f} (low = oversaturated)")
    print(f"   Starbucks within 500m: {row['n_starbucks_500m']:.0f}")
    print(f"   Subway ridership: {row['mta_avg_daily_ridership']:,.0f}/day")
    print(f"   Nearest Starbucks: {row['nearest_starbucks_dist_m']:.0f}m")
    print(f"   Closure risk score: {row['closure_risk']:.3f}")

In [ ]:
# ── closure risk map ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 12))

sc = ax.scatter(df_valid["lon"], df_valid["lat"],
                c=df_valid["closure_risk"], cmap="RdYlGn_r",
                s=40, edgecolors="grey", linewidth=0.3, alpha=0.7)
plt.colorbar(sc, ax=ax, label="Closure Risk Score", shrink=0.7)

ax.scatter(top_risk["lon"], top_risk["lat"], facecolors="none",
           edgecolors="red", s=200, linewidth=2, zorder=5,
           label="Top 5 closure risk")

for rank, (idx, row) in enumerate(top_risk.iterrows(), 1):
    ax.annotate(f"#{rank}", (row["lon"], row["lat"]),
                textcoords="offset points", xytext=(8, 8),
                fontsize=9, fontweight="bold", color="red")

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Closure Risk — Red = High Risk, Green = Safe")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## Section 4 — Competitive White Space

**White space** = areas where competitors are distant but demand signals are strong. These are "blue ocean" opportunities.

In [ ]:
# ── white space score ────────────────────────────────────────────────
df_valid["white_space"] = (
    norm(df_valid["nearest_competitor_dist_m"]) * 0.3 +
    norm(df_valid["mta_avg_daily_ridership"]) * 0.3 +
    norm(df_valid["tract_population"]) * 0.2 +
    norm(df_valid["ped_count_nearest"].fillna(0)) * 0.2
)

top_white = df_valid.nlargest(5, "white_space")

print("Top 5 Competitive White Space Opportunities:")
print("=" * 90)
for rank, (idx, row) in enumerate(top_white.iterrows(), 1):
    print(f"\n#{rank} — Lat: {row['lat']:.5f}, Lon: {row['lon']:.5f}")
    print(f"   Nearest competitor: {row['nearest_competitor_dist_m']:.0f}m")
    print(f"   Subway ridership: {row['mta_avg_daily_ridership']:,.0f}/day")
    print(f"   Tract population: {row['tract_population']:,.0f}")
    print(f"   Pedestrian count: {row['ped_count_nearest']:,.0f}")
    print(f"   White space score: {row['white_space']:.3f}")

In [ ]:
# ── combined strategic map ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 12))

ax.scatter(df_valid["lon"], df_valid["lat"], c="lightgrey", s=15,
           edgecolors="grey", linewidth=0.3, alpha=0.4, label="All stores")

ax.scatter(top_expand["lon"], top_expand["lat"], c="#D4AF37", s=200,
           edgecolors="black", linewidth=1.5, marker="*", zorder=5,
           label="Expand here")

ax.scatter(top_risk["lon"], top_risk["lat"], c="#D32F2F", s=100,
           edgecolors="black", linewidth=1, marker="X", zorder=5,
           label="Consider closing")

ax.scatter(top_white["lon"], top_white["lat"], c="#1976D2", s=120,
           edgecolors="black", linewidth=1, marker="D", zorder=5,
           label="White space")

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Strategic Location Map — Expand, Close, or Dominate")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

## Section 5 — Key Recommendations

### Market Context

Starbucks' FY2024 10-K reports **$26.5B in North American revenue** across ~16,600 US company-operated stores, implying **~$1.6M average annual revenue per store**. Manhattan's 171 stores represent roughly **$274M in annual revenue** — a significant concentration in 23 square miles.

The Manhattan cafe market includes 1,200+ competing cafes. With an estimated **$800M–$1B total addressable market** for specialty coffee in Manhattan (based on store density × average revenue benchmarks), Starbucks holds an estimated **27–34% market share** by location count.

### Actionable Recommendations

### 1. Expand near high-ridership subway stations with low Starbucks density
Subway ridership is the strongest predictor of store success (r=0.58), yet several high-ridership stations have no Starbucks within 300m. At ~$1.6M revenue per store, each new well-placed location represents significant incremental revenue. Our top 5 expansion candidates sit near stations with 25,000+ daily riders.

### 2. Consolidate in oversaturated Midtown clusters
Multiple stores in Midtown have 4+ Starbucks within 500m and declining relative demand. Closing the weakest performer in each cluster would reduce cannibalization and improve per-store economics. If consolidating 5 underperforming stores redirects even 60% of their traffic to nearby surviving stores, net revenue loss is minimal while lease costs drop by ~$1M+/year.

### 3. Target competitive white space before rivals do
We identified areas where the nearest competitor is 200m+ away despite strong foot traffic. These "blue ocean" locations offer first-mover advantage before Dunkin' or independent cafes fill the gap. With 1,200+ competitors already in Manhattan, defensible white space is a shrinking resource.

### 4. Ignore household income as a site selection factor
Our data shows near-zero correlation (r=0.03) between tract median income and location fitness. Site selection should weight transit access and pedestrian flow, not neighborhood affluence. This finding challenges conventional retail wisdom and suggests Starbucks' pricing power transcends income demographics.

*Note: Revenue estimates are derived from Starbucks' public 10-K filings and industry benchmarks. Actual per-store revenue varies significantly by location.*

## Section 6 — Limitations & Series Navigation

### Limitations

| # | Limitation | Impact |
|---|-----------|--------|
| 1 | **Point-in-time snapshot** — no store revenue or profitability data | Cannot validate whether "high LFS" actually means "high profit" |
| 2 | **Manhattan only** — results may not generalize | Suburban/other city dynamics differ significantly |
| 3 | **No demand-side ground truth** — LFS is a proxy, not actual sales | Recommendations are directional, not definitive |
| 4 | **Static competition landscape** — competitors open and close | White space opportunities may already be filled |
| 5 | **No rent/lease cost data** — a critical factor in real site selection | A location can score high on demand but be unprofitable due to rent |

### Series Navigation

| # | Notebook | Link |
|---|----------|------|
| 0 | Manhattan Café Wars — EDA | [Kaggle](https://www.kaggle.com/code/shiratoriseto/manhattan-cafe-wars-starbucks-vs-1200-competitors) |
| 1 | Starbucks 10-K NLP: Topic & Keyword Analysis | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-10-k-nlp-topic-keyword-analysis) |
| 1F | Starbucks 10-K LDA Topic Explorer (pyLDAvis) | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-10-k-lda-topic-explorer-pyldavis) |
| 2A | Starbucks Spatial Clustering | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-spatial-clustering) |
| 2B | Starbucks Location Fitness | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-location-fitness) |
| 2C | Starbucks Walk-Distance Analysis (OSMnx) | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-walk-distance-analysis-osmnx) |
| 5 | LFS Predictive Model | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-location-fitness-score-predictive-model) |
| **5B** | **Strategic Location Insights (this notebook)** | — |
| C | Data Pipeline: EDGAR + OSM → CSV | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-data-pipeline-edgar-osm-to-csv) |
| D | US Expansion Animated Choropleth | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-us-expansion-animated-choropleth) |
| G | NLP × Spatial Integration | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-nlp-x-spatial) |

**Dataset:** [shiratoriseto/manhattan-cafe-wars](https://www.kaggle.com/datasets/shiratoriseto/manhattan-cafe-wars)  
**Model:** [shiratoriseto/starbucks-location-fitness-model](https://www.kaggle.com/models/shiratoriseto/starbucks-location-fitness-model)  
**GitHub:** [seto-siratori/starbucks-kaggle](https://github.com/seto-siratori/starbucks-kaggle)